# Met Office historical data (AWS archive)

The Met Office publishes a 2-year rolling archive of the UK 2 km model (same model as the DataHub order) as a public S3 bucket — no AWS account or API key needed ([listing](https://aws.amazon.com/marketplace/pp/prodview-oiodcatwsyjwm), licence CC BY-SA 4.0).

Downloading and caching is handled by `MetOfficeLoader` in [`loaders/met_office.py`](../loaders/met_office.py); files are saved under `data/` next to this notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from datetime import datetime, timedelta
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap

# Make the project-root loaders package importable from this folder
sys.path.insert(0, str(Path.cwd().parent))
from loaders import MetOfficeLoader

In [ ]:
loader = MetOfficeLoader(data_dir="data")

In [ ]:
RUN = datetime(2025, 10, 19, 0)  # any hourly run in the last 2 years
SHOW_OUTPUT = False  # list the variables available for RUN

if SHOW_OUTPUT:
    display(loader.list_variables(RUN))

In [ ]:
SHOW_OUTPUT = False  # display the loaded rainfall array

archive = loader.load_run(RUN, steps=range(1, 7), variable="rainfall_accumulation-PT01H")

# Archive rainfall is in metres; convert to mm
archive_rain = archive["thickness_of_rainfall_amount"] * 1000
archive_rain.attrs["units"] = "mm"
if SHOW_OUTPUT:
    display(archive_rain)

In [ ]:
START = datetime(2025, 10, 19, 0)
END = datetime(2025, 10, 26, 0)
STEP = timedelta(hours=3)  # must be a whole number of hours
SHOW_OUTPUT = False  # display the loaded rainfall and wind arrays

# Each frame is the 1-hour forecast from the run issued an hour before it
rain_ds = loader.load_series(START, END, STEP, "rainfall_accumulation-PT01H")
uk_crs = loader.crs(rain_ds)
# Archive rainfall is in metres; convert to mm
rain_series = rain_ds["thickness_of_rainfall_amount"] * 1000
rain_series.attrs["units"] = "mm"

# 10 m wind: speed in m/s, and the direction it blows *from*, in degrees clockwise from north
wind_speed = loader.load_series(START, END, STEP, "wind_speed_at_10m")["wind_speed"]
wind_from_direction = loader.load_series(START, END, STEP, "wind_direction_at_10m")["wind_from_direction"]
if SHOW_OUTPUT:
    display(rain_series, wind_speed, wind_from_direction)

In [ ]:
ARROW_SPACING = 50  # draw a wind arrow every this many grid points (~2 km each)
MIN_RAIN = 0.05  # mm; lighter rain is left transparent so the land/sea colours show through
LAND_COLOUR = "#e3dcc8"
SEA_COLOUR = "#cfdde6"

# Thin the wind to a coarser grid, then convert speed/direction to eastward (u) and northward (v)
# components. Negate because the direction is where the wind blows from, not towards.
thin = {"projection_x_coordinate": slice(None, None, ARROW_SPACING),
        "projection_y_coordinate": slice(None, None, ARROW_SPACING)}
speed = wind_speed.isel(thin)
direction = np.deg2rad(wind_from_direction.isel(thin))
u = (-speed * np.sin(direction)).values
v = (-speed * np.cos(direction)).values
# u/v are relative to true north; rotate them to the map grid's x/y axes
arrow_x, arrow_y = np.meshgrid(speed.projection_x_coordinate, speed.projection_y_coordinate)
lonlat = ccrs.PlateCarree().transform_points(uk_crs, arrow_x, arrow_y)
for i in range(len(u)):
    u[i], v[i] = uk_crs.transform_vectors(ccrs.PlateCarree(), lonlat[..., 0], lonlat[..., 1], u[i], v[i])

fig, ax = plt.subplots(figsize=(7, 6), subplot_kw={"projection": uk_crs})
ax.add_feature(cfeature.LAND.with_scale("10m"), facecolor=LAND_COLOUR, zorder=0)
ax.add_feature(cfeature.OCEAN.with_scale("10m"), facecolor=SEA_COLOUR, zorder=0)
ax.coastlines(resolution="10m", linewidth=0.5)
ax.add_feature(cfeature.BORDERS.with_scale("10m"), linewidth=0.5)
# Hide near-zero rain, and start the colour map at mid-blue so light rain stands out from the sea
rain_shown = rain_series.where(rain_series >= MIN_RAIN)
rain_cmap = ListedColormap(plt.get_cmap("Blues")(np.linspace(0.35, 1, 256)))
# Fix the colour scale across all frames (98th percentile, like robust=True)
mesh = ax.pcolormesh(
    rain_shown.projection_x_coordinate, rain_shown.projection_y_coordinate, rain_shown.isel(time=0),
    transform=uk_crs, cmap=rain_cmap, vmin=MIN_RAIN, vmax=float(rain_series.quantile(0.98)),
)
fig.colorbar(mesh, ax=ax, shrink=0.8, extend="max", label="Rainfall in previous hour (mm)")
# Arrow length is proportional to wind speed, on the same scale in every frame
arrows = ax.quiver(arrow_x, arrow_y, u[0], v[0], transform=uk_crs, color="0.2",
                   scale=350, scale_units="width", width=0.0025)
ax.quiverkey(arrows, 0.88, -0.04, 10, "10 m/s wind", labelpos="W", coordinates="axes")


def draw_frame(i):
    frame = rain_shown.isel(time=i)
    mesh.set_array(np.ma.masked_invalid(frame.values))
    arrows.set_UVC(u[i], v[i])
    ax.set_title(f"{frame.time.dt.strftime('%Y-%m-%d %H:%M').item()} UTC")
    return mesh, arrows


plt.rcParams["animation.embed_limit"] = 100  # MB; the default 20 MB cuts off longer animations
anim = FuncAnimation(fig, draw_frame, frames=rain_series.sizes["time"], interval=250)
plt.close(fig)  # stop the static first frame also being shown
HTML(anim.to_jshtml())